In [2]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from scipy.stats import entropy

# 1. Load the Iris Dataset
iris = load_iris()
X = iris.data
y = iris.target
feature_idx = 2 # We will focus on Petal Length (index 2)
feature_name = iris.feature_names[feature_idx]

petal_lengths = X[:, feature_idx]

# --- Core Math Functions ---
def calc_entropy(labels):
    """Calculates Shannon Entropy for a given array of class labels."""
    if len(labels) == 0:
        return 0
    counts = np.bincount(labels)
    probs = counts[counts > 0] / len(labels) # Avoid log(0)
    return entropy(probs, base=2)

def calc_information_gain(feature_values, labels, threshold):
    """Calculates Information Gain for a specific split threshold."""
    parent_entropy = calc_entropy(labels)
    
    # Create the split
    left_mask = feature_values <= threshold
    right_mask = feature_values > threshold
    
    left_labels = labels[left_mask]
    right_labels = labels[right_mask]
    
    # Calculate weighted child entropy
    weight_left = len(left_labels) / len(labels)
    weight_right = len(right_labels) / len(labels)
    
    child_entropy = (weight_left * calc_entropy(left_labels)) + (weight_right * calc_entropy(right_labels))
    return parent_entropy - child_entropy

# --- Step-by-Step Visualization ---
# Test every unique value of Petal Length as a potential split threshold
thresholds = np.unique(petal_lengths)
info_gains = [calc_information_gain(petal_lengths, y, t) for t in thresholds]

# Find the absolute best split
best_idx = np.argmax(info_gains)
best_threshold = thresholds[best_idx]
max_gain = info_gains[best_idx]

# --- Plotting the Results ---
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

# Top Plot: The raw data distribution (jittered for visibility)
jitter = np.random.normal(0, 0.05, size=len(petal_lengths))
scatter = ax1.scatter(petal_lengths, y + jitter, c=y, cmap='viridis', edgecolor='k', s=60, alpha=0.8)
ax1.axvline(best_threshold, color='red', linestyle='--', linewidth=2, label=f'Optimal Split ({best_threshold}cm)')
ax1.set_ylabel('Species (Class)')
ax1.set_title(f'Distribution of {feature_name} by Species')
ax1.set_yticks([0, 1, 2])
ax1.set_yticklabels(iris.target_names)
ax1.legend()

# Bottom Plot: The Information Gain Curve
ax2.plot(thresholds, info_gains, marker='o', linestyle='-', color='b')
ax2.axvline(best_threshold, color='red', linestyle='--', linewidth=2)
ax2.scatter([best_threshold], [max_gain], color='red', s=100, zorder=5) # Highlight max
ax2.set_xlabel(f'{feature_name} (cm)')
ax2.set_ylabel('Information Gain (Bits)')
ax2.set_title('Information Gain vs. Split Threshold')
ax2.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

print(f"Parent (Root) Entropy: {calc_entropy(y):.3f} bits")
print(f"Best Split Threshold for {feature_name}: <= {best_threshold} cm")
print(f"Maximum Information Gain Achieved: {max_gain:.3f} bits")

Parent (Root) Entropy: 1.585 bits
Best Split Threshold for petal length (cm): <= 1.9 cm
Maximum Information Gain Achieved: 0.918 bits
